In [ ]:
# Kachchh Dust & Heat Stress Dataset Downloader (PS000109)
# Automatically fetches hourly PM10, PM2.5, Mineral Dust, and WBGT Heat Stress parameters across 5 Kachchh zones

import requests
import json
import csv
import math
from datetime import datetime

ZONES = {
    "mundra_port": {"name": "Mundra Port & Industrial Hub", "lat": 22.74, "lon": 69.70},
    "kandla_port": {"name": "Kandla Port & Gandhidham", "lat": 23.03, "lon": 70.22},
    "bhuj_inland": {"name": "Bhuj District Capital & Inland Mining", "lat": 23.25, "lon": 69.67},
    "little_rann_saltpan": {"name": "Little Rann Salt Pans (Agariyas)", "lat": 23.35, "lon": 70.70},
    "abdasa_belt": {"name": "Abdasa Lignite & Cement Belt", "lat": 23.43, "lon": 68.85}
}

def calculate_outdoor_wbgt(temp_c, rel_humidity, wind_speed_kmh, direct_irradiance):
    """
    Approximates ISO 7243 Outdoor WBGT = 0.7*Tnw + 0.2*Tg + 0.1*Tdb
    """
    # Stull formula for wet-bulb temperature
    t = temp_c
    rh = rel_humidity
    tw = (t * math.atan(0.151977 * math.sqrt(rh + 8.313659)) 
          + math.atan(t + rh) 
          - math.atan(rh - 1.676331) 
          + 0.00391838 * (rh ** 1.5) * math.atan(0.023101 * rh) 
          - 4.686035)
    
    # Globe temperature approximation based on solar irradiance and wind
    wind_ms = max(wind_speed_kmh / 3.6, 0.5)
    tg = t + (0.015 * direct_irradiance) / (math.sqrt(wind_ms))
    
    # ISO 7243 Outdoor WBGT
    wbgt = 0.7 * tw + 0.2 * tg + 0.1 * t
    return round(wbgt, 2)

print("Kachchh Dataset Ingestion Engine Initialized successfully!")


In [ ]:
# 2. Industrial Job-Exposure Matrix (JEM) for Kachchh (PS000109)
# Maps job roles, exposure limits, and statutory medical screening schedules under OSH Code 2020

jem_data = [
    {
        "job_code": "JEM-KCH-01",
        "job_title": "Salt-pan Worker (Agariya)",
        "worksite_type": "Salt Pans (Little & Great Rann)",
        "primary_hazard": "Extreme Solar WBGT Heat Stress & Brine Glare",
        "typical_shift_hours": 10.0,
        "pel_wbgt_celsius": 28.0,
        "pel_dust_pm10_ugm3": 150.0,
        "trigger_condition": "WBGT > 28°C for >= 5 consecutive shifts",
        "statutory_clinical_test": "Urine Specific Gravity, Serum Creatinine & eGFR (Renal Check)",
        "statutory_frequency": "Every 30 days during harvesting season",
        "regulatory_ref": "OSH Code 2020 Third Schedule, Gujarat Factories Rules 2025"
    },
    {
        "job_code": "JEM-KCH-02",
        "job_title": "Cement Packing & Bagging Operator",
        "worksite_type": "Cement Manufacturing Plants (Mundra / Abdasa)",
        "primary_hazard": "Respirable Crystalline Silica & Clinker Dust",
        "typical_shift_hours": 8.0,
        "pel_wbgt_celsius": 30.0,
        "pel_dust_pm10_ugm3": 50.0,
        "trigger_condition": "Cumulative PM10 > 50 ug/m3 8h-TWA for >= 20 shifts",
        "statutory_clinical_test": "Spirometry (FEV1, FVC) & Posterior-Anterior Chest X-Ray (ILO)",
        "statutory_frequency": "Every 6 months (Mandatory under Sec 41C Factories Act)",
        "regulatory_ref": "Factories Act 1948 Sec 41C, Notifiable Disease #12 Silicosis"
    },
    {
        "job_code": "JEM-KCH-03",
        "job_title": "Port Container Yard & Crane Operator",
        "worksite_type": "Mundra / Deendayal (Kandla) Port Yards",
        "primary_hazard": "Radiant Steel Heat, Diesel Particulate Matter (DPM)",
        "typical_shift_hours": 12.0,
        "pel_wbgt_celsius": 29.0,
        "pel_dust_pm10_ugm3": 100.0,
        "trigger_condition": "WBGT > 29°C for >= 4 shifts OR High DPM exposure",
        "statutory_clinical_test": "Cardiovascular Stress Test, Hydration & Peak Expiratory Flow",
        "statutory_frequency": "Annually or exposure-triggered within 7 days",
        "regulatory_ref": "Dock Workers (Safety, Health and Welfare) Act & OSH Code 2020"
    },
    {
        "job_code": "JEM-KCH-04",
        "job_title": "Heavy Mineral Tipper / Truck Driver",
        "worksite_type": "Kachchh Highway & Port Corridors",
        "primary_hazard": "Whole-Body Vibration, High Cabin Heat, Road Silica Dust",
        "typical_shift_hours": 12.0,
        "pel_wbgt_celsius": 31.0,
        "pel_dust_pm10_ugm3": 120.0,
        "trigger_condition": "Shift duration > 10h in ambient heat > 40°C",
        "statutory_clinical_test": "Spinal Health Audit, Blood Pressure & Hydration Index",
        "statutory_frequency": "Semi-annually",
        "regulatory_ref": "Motor Transport Workers Act / OSH Code 2020"
    },
    {
        "job_code": "JEM-KCH-05",
        "job_title": "Stone Crusher & Mineral Grinding Operator",
        "worksite_type": "Inland Crushing Units (Bhuj / Anjar)",
        "primary_hazard": "Acute High-Concentration Quartz Silica Dust (SiO2 > 30%)",
        "typical_shift_hours": 8.0,
        "pel_wbgt_celsius": 30.0,
        "pel_dust_pm10_ugm3": 25.0,
        "trigger_condition": "Cumulative respirable silica > 0.05 mg/m3",
        "statutory_clinical_test": "Digital Chest Radiography (B-reader ILO) + Full Pulmonary Function",
        "statutory_frequency": "Every 3 months (High-Risk Category)",
        "regulatory_ref": "National Institute of Occupational Health (NIOH) Silicosis Guidelines"
    }
]

print(f"Loaded {len(jem_data)} Job-Exposure Matrix entries!")
